# 延長確認シート作成

Salesforce の稼働データ（`kado__c`）から延長確認の対象者を抽出し、Google スプレッドシートへ出力します。

## 日付の選び方（重要）

**10/1〜の延長を出す場合は、2か月前の日付を選択してください。**

2026/10/1 からの延長を確認する場合は、**2026/08/01** と 2か月前で入力していただければ対象者が出力されます。

※ **年月のみ使用**していますので、日付（日）はいくつでもOKです。  
　例: `2026/08/01` でも `2026/08/15` でも、どちらも「2026年8月」として扱います。

| 確認したい延長開始 | Colab で選ぶ日付 | 抽出される延長確認年月 | 出力シート名 |
|---|---|---|---|
| 2026/10/1〜 | 2026/08/01（2か月前） | 2026年08月 | `202608` |
| 2026/11/1〜 | 2026/09/01（2か月前） | 2026年09月 | `202609` |
| 2027/01/1〜 | 2026/11/01（2か月前） | 2026年11月 | `202611` |

## 実行手順

1. Colab のシークレット（🔑）に以下を設定する
   - `SF_CLIENT_ID` … Salesforce の Client ID
   - `SF_USERNAME` … Salesforce のユーザー名
   - `SF_PRIVATE_KEY` … JWT 署名用の秘密鍵（PEM）
2. 日付フォームで **延長開始月の 2か月前** を選ぶ
3. フォーム下のセルを実行する（または「ランタイム」→「すべてのセルを実行」）
4. Google アカウント認証が出たら許可する

## 取得条件

- `entyokakuninnengetsu__c` と `kadonengetsu__c` の両方が、選んだ日付の年月と一致する `kado__c` を取得します
- 契約更新日は親契約（`oyakeiyakuid__c` → `keiyaku__c.keiyakukoushinbi__c`）から取得し、シートの R 列へ出力します
- 営アシ対応ストップは「BO延長確認停止（上案件）」「BO延長確認停止（下案件）」を、上案件・下案件の 2 列で出力します
- 最終列に稼働レコードの ID（`レコードID`）を出力します。Salesforce への書き戻しに使います

## 出力先

毎回、次のスプレッドシートを更新します。

[延長確認シートを開く](https://docs.google.com/spreadsheets/d/13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk/edit)

https://docs.google.com/spreadsheets/d/13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk/edit

対象月のタブ（例: `202608`）が無ければ、その名前で新規作成します。


## 事前準備

初回のみ、Colab 左の 🔑 **シークレット** に Salesforce 認証情報を入れてください。

実行すると Google アカウントへのアクセス許可を求められます。スプレッドシート出力に必要なので許可してください。


In [1]:
# @title 延長確認シート作成
# 延長開始月の2か月前の日付を指定（例: 2026/10/1開始 → 2026/08/01）
# ※ 年月のみ使用するため、日付（日）はいくつでもOKです
_sheetname_input = "2026-10-01"  # @param {"type":"date"}

!pip install simple-salesforce gspread pandas
!pip install pyjwt cryptography requests

import requests
import pandas as pd
import gspread
from google.colab import userdata, auth
from google.auth import default
from simple_salesforce import Salesforce
import json
from datetime import datetime, date
import time
import jwt

# Salesforce 延長確認年月フィールド（抽出条件）
ENTYOKAKUNINNENGETSU_FIELD = "entyokakuninnengetsu__c"
ENTYOKAKUNINNENGETSU_LABEL = "延長確認年月"
KADONENGETSU_FIELD = "kadonengetsu__c"
# 親契約（keiyaku__c）参照
OYAKEIYAKU_ID_FIELD = "oyakeiyakuid__c"
KEIYAKU_KOUSHINBI_FIELD = "keiyakukoushinbi__c"
KEIYAKU_KOUSHINBI_LABEL = "契約更新日"
ARARI_COLUMN_LABEL = "粗利"
RYUNYUMOTO_COLUMN_LABEL = "流入元"
UE_STOP_FIELD_LABEL = "BO延長確認停止（上案件）"
SHITA_STOP_FIELD_LABEL = "BO延長確認停止（下案件）"
UE_EIASHI_STOP_LABEL = "上案件\nバックオフィスからの\n対応ストップ案件チェック"
SHITA_EIASHI_STOP_LABEL = "下案件\nバックオフィスからの\n対応ストップ案件チェック"
RECORD_ID_LABEL = "レコードID"
OUTPUT_SPREADSHEET_ID = "13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk"
OUTPUT_SPREADSHEET_URL = (
    "https://docs.google.com/spreadsheets/d/" + OUTPUT_SPREADSHEET_ID + "/edit"
)
ARARI_RF_THRESHOLD = 100_000
ARARI_HIGHLIGHT_COLOR = {"red": 0.96, "green": 0.6, "blue": 0.6}
UE_KOUSHIN_ISHI_LABEL = "㊤更新意思"
SHITA_KOUSHIN_ISHI_LABEL = "㊦更新意思"
KOUSHIN_ISHI_BACKGROUND_COLOR = {"red": 0.82, "green": 0.94, "blue": 0.98}
KEIYAKU_SYURYO_LABEL = "契約終了理由"
TAJIAN_KEIYAKU_SYURYO_LABEL = "他案件契約終了日\n（㊦担当者入力）"
JOKEN_HENKO_LABEL = "条件変更の有無\n確認したかチェック（全件）"
JOKEN_HENKO_DROPDOWN_OPTIONS = ["無し", "有り"]
KEIZOKU_MIKOMI_CHECK_LABEL = "（仮）継続見込みチェック"
YOJITSU_CHECK_LABEL = "予実\n①全体稼働\n②注文書"
GETSUMATSU_JOKYO_CHECK_LABEL = "月末時\n状況確認\nの有無"
MANUAL_CHECKBOX_COLUMN_LABELS = [
    KEIZOKU_MIKOMI_CHECK_LABEL,
    YOJITSU_CHECK_LABEL,
    GETSUMATSU_JOKYO_CHECK_LABEL,
]
KEIYAKU_SYURYO_DROPDOWN_OPTIONS = [
    "再販不可/勤怠不良",
    "再販不可/人物NG",
    "再販不可/スキル不足",
    "再販不可/別案件が既に確定",
    "再販不可/フリーランス辞める",
    "再販不可/ROSCAが嫌われた",
    "再販不可/エンジニア辞める",
    "再販不可/その他",
    "再販可/プロジェクト終了/縮小",
    "再販可/本人希望",
    "再販可/スキル不足",
    "再販可/勤怠不良",
    "再販可/その他",
]

def get_salesforce_access_token():
    """
    Salesforceのアクセストークンを取得
    """
    try:
        print("🔐 アクセストークンを取得中...")
        sf_client_id = userdata.get('SF_CLIENT_ID')
        sf_username = userdata.get('SF_USERNAME')
        sf_private_key = userdata.get('SF_PRIVATE_KEY')
        if not sf_client_id or not sf_username or not sf_private_key:
            raise RuntimeError("Salesforce シークレット (SF_CLIENT_ID / SF_USERNAME / SF_PRIVATE_KEY) が見つかりません。")

        normalized_key = str(sf_private_key).strip().replace("\\n", "\n")
        claims = {
            "iss": sf_client_id,
            "sub": sf_username,
            "aud": "https://login.salesforce.com",
            "exp": int(time.time()) + 180,
        }
        assertion = jwt.encode(claims, normalized_key, algorithm="RS256")

        res = requests.post("https://login.salesforce.com/services/oauth2/token", data={
            "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
            "assertion": assertion
        })

        if res.status_code != 200:
            print(f"❌ 認証エラー: HTTP {res.status_code}")
            print(f"レスポンス: {res.text}")
            raise Exception(f"認証に失敗しました: {res.status_code}")

        token_data = res.json()

        if 'access_token' not in token_data:
            print(f"❌ アクセストークンが取得できませんでした: {token_data}")
            raise Exception("アクセストークンが含まれていません")
        if 'instance_url' not in token_data:
            print(f"❌ instance_url が取得できませんでした: {token_data}")
            raise Exception("instance_url が含まれていません")

        print("✓ アクセストークン取得完了")
        return token_data["access_token"], token_data["instance_url"]

    except Exception as e:
        print(f"❌ アクセストークン取得エラー: {e}")
        print("💡 以下を確認してください:")
        print("   - SF_CLIENT_ID (Client ID)")
        print("   - SF_USERNAME (Salesforce Username)")
        print("   - SF_PRIVATE_KEY (JWT Private Key)")
        raise

def setup_salesforce_connection():
    """
    Salesforce接続を設定
    """
    try:
        # SalesforceのOAuth(JWT)で取得した情報
        access_token, instance_url = get_salesforce_access_token()
        print(f"🌐 接続先: {instance_url}")

        sf = Salesforce(instance_url=instance_url, session_id=access_token)
        print("✓ Salesforce接続オブジェクト作成完了")
        return sf

    except Exception as e:
        print(f"❌ Salesforce接続エラー: {e}")
        raise

def _parse_sf_date(value):
    """Salesforceの日付/日時文字列を date に変換する"""
    if value is None or value == "":
        return None
    if isinstance(value, date):
        return value
    if isinstance(value, datetime):
        return value.date()
    text = str(value).strip()
    if not text:
        return None
    try:
        return datetime.fromisoformat(text.replace("Z", "+00:00")).date()
    except ValueError:
        pass
    try:
        return datetime.strptime(text[:10], "%Y-%m-%d").date()
    except ValueError:
        return None


def _parse_year_month_from_sheetname(value):
    """文字列（例: "202605"）から年月を抽出する"""
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None

    if len(text) == 6 and text.isdigit():
        year = int(text[:4])
        month = int(text[4:6])
        if 1 <= month <= 12:
            return year, month

    for sep in ("-", "/", "."):
        if sep in text:
            parts = text.split(sep)
            if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
                year = int(parts[0])
                month = int(parts[1])
                if 1 <= month <= 12:
                    return year, month

    return None


def _parse_period_value(value):
    """
    日付型・文字列から (sheetname_YYYYMM, year, month) を返す。
    Colab @param type:date および従来の "202605" 文字列に対応。
    """
    if value is None:
        return None

    if isinstance(value, datetime):
        d = value.date()
        return f"{d.year:04d}{d.month:02d}", d.year, d.month

    if isinstance(value, date):
        return f"{value.year:04d}{value.month:02d}", value.year, value.month

    text = str(value).strip()
    if not text:
        return None

    ym = _parse_year_month_from_sheetname(text)
    if ym:
        year, month = ym
        return f"{year:04d}{month:02d}", year, month

    d = _parse_sf_date(text)
    if d:
        return f"{d.year:04d}{d.month:02d}", d.year, d.month

    return None


def _escape_soql_literal(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def _format_entyokakunin_nengetsu(year, month):
    """年月の標準形式（例: 2026年01月）"""
    return f"{int(year):04d}年{int(month):02d}月"


def _nengetsu_match_candidates(year, month):
    """年月文字列の照合候補を生成する"""
    y, m = int(year), int(month)
    return list(dict.fromkeys([
        _format_entyokakunin_nengetsu(y, m),
        f"{y:04d}年{m}月",
    ]))


def _build_nengetsu_filter(field_name, year, month):
    """年月文字列項目の SOQL 条件を組み立てる"""
    candidates = _nengetsu_match_candidates(year, month)
    clauses = [
        f"{field_name} = '{_escape_soql_literal(c)}'"
        for c in candidates
    ]
    return "(" + " OR ".join(clauses) + ")"


def _build_kado_where_filter(year, month):
    """entyokakuninnengetsu__c と kadonengetsu__c が指定年月と一致する条件"""
    return (
        _build_nengetsu_filter(ENTYOKAKUNINNENGETSU_FIELD, year, month)
        + " AND "
        + _build_nengetsu_filter(KADONENGETSU_FIELD, year, month)
    )


def _log_entyokakunin_diagnostics(sf, year, month):
    """0件時に年月項目の実データ形式を表示する"""
    diag_soql = (
        f"SELECT {ENTYOKAKUNINNENGETSU_FIELD}, {KADONENGETSU_FIELD} FROM kado__c "
        f"WHERE {ENTYOKAKUNINNENGETSU_FIELD} != null "
        "LIMIT 30"
    )
    try:
        result = sf.query_all(diag_soql)
        records = result.get("records", []) or []
        entyo_values = [r.get(ENTYOKAKUNINNENGETSU_FIELD) for r in records]
        kado_values = [r.get(KADONENGETSU_FIELD) for r in records]
        print(f"💡 参考: {ENTYOKAKUNINNENGETSU_FIELD} サンプル: {entyo_values}")
        print(f"💡 参考: {KADONENGETSU_FIELD} サンプル: {kado_values}")
        print(f"💡 今回の照合候補: {_nengetsu_match_candidates(year, month)}")
    except Exception as e:
        print(f"💡 診断クエリ失敗: {e}")


def _resolve_sheetname_config():
    """
    sheetname（ワークシート名 YYYYMM）と対象年月を解決する。
    日付フォーム（_sheetname_input）を優先。未解釈時のみ既存 sheetname を使用。
    """
    global sheetname

    parsed = _parse_period_value(_sheetname_input)
    if parsed:
        sheetname, year, month = parsed
        print(
            f"ℹ️  日付フォーム '{_sheetname_input}' → シート名={sheetname}, 対象={year}年{month}月"
        )
        return sheetname, year, month

    if "sheetname" in globals() and globals().get("sheetname") not in (None, ""):
        source = globals().get("sheetname")
        parsed = _parse_period_value(source)
        if parsed:
            sheetname, year, month = parsed
            print(f"ℹ️  sheetname '{source}' → シート名={sheetname}, 対象={year}年{month}月")
            return sheetname, year, month

    now = datetime.now()
    sheetname = f"{now.year:04d}{now.month:02d}"
    print(f"ℹ️  年月を解釈できなかったため当月を使用: {now.year}年{now.month}月")
    return sheetname, now.year, now.month


def _resolve_target_period():
    """対象年月を返す。target_year/month 明示時のみそちらを優先。"""
    year = globals().get("target_year")
    month = globals().get("target_month")
    if year is not None and month is not None:
        return int(year), int(month)

    _, year, month = _resolve_sheetname_config()
    return year, month


def _resolve_shitakigyomei_display(gizyutusyamei, shitakigyomei):
    """下企業名: shitakigyomei__c があればそれを、空欄なら gizyutusyamei__c を返す"""
    shita = str(shitakigyomei or "").strip()
    if shita:
        return shita
    return str(gizyutusyamei or "").strip()


def _pick_final_end_date(row):
    """
    kadosakusei_loop__c が「最終」のレコードから最終終了日を取得する。
    上工事終日・下工事終日のうち存在する方を優先（両方あれば下工事終日）。
    """
    if str(row.get("kadosakusei_loop__c") or "").strip() != "最終":
        return None
    shita = _parse_sf_date(row.get("shitakoushinishi__c"))
    ue = _parse_sf_date(row.get("uekoushinishi__c"))
    return shita or ue


def _sf_id_lookup_keys(value):
    """Salesforce ID の 15桁/18桁ゆれを吸収する照合キー"""
    text = str(value or "").strip()
    if not text:
        return []
    keys = [text]
    if len(text) >= 15:
        keys.append(text[:15])
    return list(dict.fromkeys(keys))


def _format_keiyaku_koushinbi(value):
    parsed = _parse_sf_date(value)
    if parsed:
        return parsed.isoformat()
    if value is None:
        return ""
    return str(value).strip()


def _fetch_keiyaku_koushinbi_map(sf, keiyaku_ids):
    """oyakeiyakuid__c に対応する keiyaku__c.keiyakukoushinbi__c を取得する"""
    unique_ids = []
    for keiyaku_id in keiyaku_ids:
        text = str(keiyaku_id or "").strip()
        if text and text not in unique_ids:
            unique_ids.append(text)
    if not unique_ids:
        return {}

    mapping = {}
    fetched = 0
    chunk_size = 200
    for start in range(0, len(unique_ids), chunk_size):
        chunk = unique_ids[start:start + chunk_size]
        id_list = ",".join(f"'{_escape_soql_literal(x)}'" for x in chunk)
        soql = (
            f"SELECT Id, {KEIYAKU_KOUSHINBI_FIELD} FROM keiyaku__c "
            f"WHERE Id IN ({id_list})"
        )
        print(f"🔍 keiyaku__c SOQL: {soql}")
        result = sf.query_all(soql)
        records = result.get("records", []) or []
        fetched += len(records)
        for rec in records:
            rec_id = rec.get("Id")
            value = _format_keiyaku_koushinbi(rec.get(KEIYAKU_KOUSHINBI_FIELD))
            for key in _sf_id_lookup_keys(rec_id):
                mapping[key] = value
    print(f"✓ keiyaku__c から契約更新日を {fetched} 件取得")
    return mapping


def _get_kado_describe_fields(sf):
    """kado__c の describe 結果（fields）を返す"""
    describe = sf.kado__c.describe()
    return describe.get("fields", []) or []


def _find_kado_field_name_by_label(described, label):
    """項目ラベルが一致する kado__c の API 名を返す"""
    target = label.replace(" ", "").replace("\n", "")
    matches = []
    for field in described:
        name = field.get("name")
        field_label = str(field.get("label") or "").strip()
        normalized = field_label.replace(" ", "").replace("\n", "")
        if name and normalized == target:
            matches.append(name)
    if len(matches) == 1:
        return matches[0]
    hints = sorted({
        str(field.get("label") or "").strip()
        for field in described
        if any(word in str(field.get("label") or "") for word in ("ストップ", "営アシ", "延長確認停止"))
    })
    hint = "、".join(item for item in hints if item) or "(該当なし)"
    raise RuntimeError(
        f"kado__c に項目ラベル「{label}」が見つかりません。近い項目: {hint}"
    )


def _resolve_kado_select_fields(sf, requested_fields):
    """
    kado__c に実在し、参照権限がある項目だけを SELECT 対象にする。
    arari__c が無い場合はラベルに「粗利」を含む項目へ読み替える。

    Returns:
        (select_fields, internal_keys)
        select_fields: SOQL に使う API 名
        internal_keys: 各 select_fields に対応する内部キー（column_labels 用）
    """
    described = _get_kado_describe_fields(sf)
    names = {f.get("name") for f in described if f.get("name")}
    label_to_name = []
    for f in described:
        name = f.get("name")
        label = str(f.get("label") or "").strip()
        if name and label:
            label_to_name.append((label, name))

    select_fields = []
    internal_keys = []
    for field in requested_fields:
        actual = field
        if field not in names:
            if field == "arari__c":
                actual = None
                for label, name in label_to_name:
                    if "粗利" in label.replace(" ", ""):
                        actual = name
                        print(
                            f"ℹ️  粗利: 'arari__c' が無いため '{name}'（ラベル: {label}）を使用"
                        )
                        break
                if actual is None:
                    print(
                        "⚠️  粗利に相当する項目が見つからないためスキップします"
                        "（API名または項目権限を確認してください）"
                    )
                    continue
            else:
                print(f"⚠️  項目 '{field}' は kado__c に存在しないためスキップします")
                continue
        select_fields.append(actual)
        internal_keys.append(field)
    return select_fields, internal_keys


def get_salesforce_report_data(sf, report_id=None, target_year=None, target_month=None):
    """
    entyokakuninnengetsu__c と kadonengetsu__c が指定年月と一致する kado__c を取得する。

    Args:
        sf: Salesforce接続オブジェクト
        report_id: 互換用の未使用引数
        target_year: 対象年
        target_month: 対象月

    Returns:
        DataFrame: 取得データ
    """
    try:
        if target_year is None or target_month is None:
            target_year, target_month = _resolve_target_period()
        else:
            target_year = int(target_year)
            target_month = int(target_month)

        print(
            f"📅 対象年月: {target_year}年{target_month}月"
            f"（{_format_entyokakunin_nengetsu(target_year, target_month)}）"
        )

        described_fields = _get_kado_describe_fields(sf)
        ue_stop_field = _find_kado_field_name_by_label(
            described_fields, UE_STOP_FIELD_LABEL
        )
        shita_stop_field = _find_kado_field_name_by_label(
            described_fields, SHITA_STOP_FIELD_LABEL
        )

        requested_fields = [
            "Name",
            "gizyutusyamei__c",
            "uekigyomei__c",
            "uetanto__c",
            "shitakigyomei__c",
            "shitatanto__c",
            "arari__c",
            ue_stop_field,
            shita_stop_field,
            "uekoushinishi__c",
            "shitakoushinishi__c",
            "kauteikikan__c",
            "keiyakusyuryoriyu__c",
            OYAKEIYAKU_ID_FIELD,
            ENTYOKAKUNINNENGETSU_FIELD,
            "kadosakusei_loop__c",
            "ryunyumoto__c",
            "Id",
        ]
        fields, field_keys = _resolve_kado_select_fields(sf, requested_fields)
        if not fields:
            raise RuntimeError("kado__c から取得可能な項目がありません。")

        # 延長確認年月・稼働年月は「2026年01月」形式の文字列項目
        kado_where = _build_kado_where_filter(target_year, target_month)
        soql = (
            "SELECT "
            + ",".join(fields)
            + " FROM kado__c"
            + " WHERE " + kado_where
        )
        print(f"🔍 SOQL: {soql}")
        query_result = sf.query_all(soql)
        rows = query_result.get("records", []) or []

        if not rows:
            _log_entyokakunin_diagnostics(sf, target_year, target_month)

        rows_data = []
        for row in rows:
            row_data = {
                key: row.get(api_name, "")
                for api_name, key in zip(fields, field_keys)
            }
            row_data["shitakigyomei__c"] = _resolve_shitakigyomei_display(
                row_data.get("gizyutusyamei__c"),
                row_data.get("shitakigyomei__c"),
            )
            final_end_date = _pick_final_end_date(row_data)
            record_id = str(row_data.pop("Id", "") or row.get("Id") or "").strip()
            row_data["最終終了日"] = final_end_date.isoformat() if final_end_date else ""
            row_data["Id"] = record_id
            rows_data.append(row_data)

        koushinbi_map = _fetch_keiyaku_koushinbi_map(
            sf,
            [r.get(OYAKEIYAKU_ID_FIELD) for r in rows_data],
        )
        replaced_rows = []
        for row_data in rows_data:
            keiyaku_id = str(row_data.get(OYAKEIYAKU_ID_FIELD) or "").strip()
            koushinbi = ""
            for key in _sf_id_lookup_keys(keiyaku_id):
                if key in koushinbi_map:
                    koushinbi = koushinbi_map[key]
                    break
            ordered = {}
            for key, value in row_data.items():
                if key == OYAKEIYAKU_ID_FIELD:
                    ordered[KEIYAKU_KOUSHINBI_FIELD] = koushinbi
                else:
                    ordered[key] = value
            if KEIYAKU_KOUSHINBI_FIELD not in ordered:
                ordered[KEIYAKU_KOUSHINBI_FIELD] = koushinbi
            replaced_rows.append(ordered)
        rows_data = replaced_rows

        df = pd.DataFrame(rows_data)

        column_labels = {
            "Name": "レコード名",
            "gizyutusyamei__c": "技術者名",
            "uekigyomei__c": "上企業名",
            "uetanto__c": "上担当",
            "shitakigyomei__c": "下企業名",
            "shitatanto__c": "下担当",
            "arari__c": "粗利",
            ue_stop_field: UE_EIASHI_STOP_LABEL,
            shita_stop_field: SHITA_EIASHI_STOP_LABEL,
            "uekoushinishi__c": "㊤更新意思",
            "shitakoushinishi__c": "㊦更新意思",
            "kauteikikan__c": "確定期間",
            "keiyakusyuryoriyu__c": KEIYAKU_SYURYO_LABEL,
            KEIYAKU_KOUSHINBI_FIELD: KEIYAKU_KOUSHINBI_LABEL,
            ENTYOKAKUNINNENGETSU_FIELD: ENTYOKAKUNINNENGETSU_LABEL,
            "kadosakusei_loop__c": "稼働作成ループ",
            "最終終了日": "最終終了日",
            "ryunyumoto__c": RYUNYUMOTO_COLUMN_LABEL,
            "Id": RECORD_ID_LABEL,
        }
        df = df.rename(columns=column_labels)

        koushinbi_idx = df.columns.get_loc(KEIYAKU_KOUSHINBI_LABEL)
        df.insert(koushinbi_idx, TAJIAN_KEIYAKU_SYURYO_LABEL, "")
        koushinbi_idx = df.columns.get_loc(KEIYAKU_KOUSHINBI_LABEL)
        df.insert(koushinbi_idx, JOKEN_HENKO_LABEL, "")
        for label in MANUAL_CHECKBOX_COLUMN_LABELS:
            koushinbi_idx = df.columns.get_loc(KEIYAKU_KOUSHINBI_LABEL)
            df.insert(koushinbi_idx, label, False)

        return df

    except Exception as e:
        print(f"kado__c データ取得エラー: {e}")
        return pd.DataFrame()

def _column_index_to_a1_letter(column_index):
    """1始まりの列番号を列記号に変換する"""
    letters = ""
    index = column_index
    while index:
        index, remainder = divmod(index - 1, 26)
        letters = chr(65 + remainder) + letters
    return letters


def _get_sheet_conditional_format_rule_count(spreadsheet, sheet_id):
    metadata = spreadsheet.fetch_sheet_metadata()
    for sheet in metadata.get("sheets", []):
        if sheet["properties"]["sheetId"] == sheet_id:
            return len(sheet.get("conditionalFormats", []))
    return 0


def _clear_worksheet_conditional_format_rules(spreadsheet, sheet_id):
    rule_count = _get_sheet_conditional_format_rule_count(spreadsheet, sheet_id)
    if rule_count == 0:
        return
    requests = [
        {"deleteConditionalFormatRule": {"sheetId": sheet_id, "index": index}}
        for index in range(rule_count - 1, -1, -1)
    ]
    spreadsheet.batch_update({"requests": requests})


def _hide_worksheet_column(spreadsheet, sheet_id, column_index):
    """0始まりの列インデックスを非表示にする"""
    spreadsheet.batch_update({
        "requests": [{
            "updateDimensionProperties": {
                "range": {
                    "sheetId": sheet_id,
                    "dimension": "COLUMNS",
                    "startIndex": column_index,
                    "endIndex": column_index + 1,
                },
                "properties": {"hiddenByUser": True},
                "fields": "hiddenByUser",
            }
        }]
    })


def _build_boolean_conditional_format_request(
    sheet_id, col_idx, end_row, formula, background_color, index
):
    """条件付き書式ルール追加リクエストを生成する"""
    return {
        "addConditionalFormatRule": {
            "rule": {
                "ranges": [{
                    "sheetId": sheet_id,
                    "startRowIndex": 1,
                    "endRowIndex": end_row,
                    "startColumnIndex": col_idx,
                    "endColumnIndex": col_idx + 1,
                }],
                "booleanRule": {
                    "condition": {
                        "type": "CUSTOM_FORMULA",
                        "values": [{"userEnteredValue": formula}],
                    },
                    "format": {
                        "backgroundColor": background_color,
                    },
                },
            },
            "index": index,
        }
    }


def _apply_conditional_formatting(spreadsheet, worksheet, df):
    """ワークシートの条件付き書式を一括適用する"""
    if df.empty:
        return

    sheet_id = worksheet.id
    end_row = _get_data_row_end_index(df)
    _clear_worksheet_conditional_format_rules(spreadsheet, sheet_id)

    requests = []
    rule_index = 0

    if JOKEN_HENKO_LABEL in df.columns:
        joken_col_idx = list(df.columns).index(JOKEN_HENKO_LABEL)
        joken_letter = _column_index_to_a1_letter(joken_col_idx + 1)
        joken_formula = f"=LEN(TRIM({joken_letter}2))=0"
        requests.append(_build_boolean_conditional_format_request(
            sheet_id,
            joken_col_idx,
            end_row,
            joken_formula,
            ARARI_HIGHLIGHT_COLOR,
            rule_index,
        ))
        rule_index += 1

    if ARARI_COLUMN_LABEL in df.columns and RYUNYUMOTO_COLUMN_LABEL in df.columns:
        arari_col_idx = list(df.columns).index(ARARI_COLUMN_LABEL)
        ryun_col_idx = list(df.columns).index(RYUNYUMOTO_COLUMN_LABEL)
        arari_letter = _column_index_to_a1_letter(arari_col_idx + 1)
        ryun_letter = _column_index_to_a1_letter(ryun_col_idx + 1)

        _hide_worksheet_column(spreadsheet, sheet_id, ryun_col_idx)

        arari_formula = f'=AND(${ryun_letter}2="RF",{arari_letter}2<{ARARI_RF_THRESHOLD})'
        requests.append(_build_boolean_conditional_format_request(
            sheet_id,
            arari_col_idx,
            end_row,
            arari_formula,
            ARARI_HIGHLIGHT_COLOR,
            rule_index,
        ))

    if requests:
        spreadsheet.batch_update({"requests": requests})
        print(f"✓ 条件付き書式を適用しました（{end_row - 1}行）")


def _apply_koushin_ishi_background(spreadsheet, worksheet, df):
    """㊤更新意思・㊦更新意思列のデータ行（ヘッダ除く）に薄水色の背景を設定する"""
    if df.empty:
        return

    sheet_id = worksheet.id
    end_row = _get_data_row_end_index(df)
    requests = []

    for label in (UE_KOUSHIN_ISHI_LABEL, SHITA_KOUSHIN_ISHI_LABEL):
        if label not in df.columns:
            continue
        col_idx = list(df.columns).index(label)
        requests.append({
            "repeatCell": {
                "range": {
                    "sheetId": sheet_id,
                    "startRowIndex": 1,
                    "endRowIndex": end_row,
                    "startColumnIndex": col_idx,
                    "endColumnIndex": col_idx + 1,
                },
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": KOUSHIN_ISHI_BACKGROUND_COLOR,
                    }
                },
                "fields": "userEnteredFormat.backgroundColor",
            }
        })

    if requests:
        spreadsheet.batch_update({"requests": requests})
        print(f"✓ ㊤更新意思・㊦更新意思列に薄水色の背景を適用しました（{end_row - 1}行）")


def _get_data_row_end_index(df):
    """データ行の終端（0始まり・排他的）を返す。ヘッダ行は含まない"""
    return len(df) + 1


def _worksheet_size_for_df(df, row_buffer=2):
    """DataFrame に合わせたシートの行数・列数を返す"""
    rows = max(len(df) + 1 + row_buffer, 2)
    cols = max(len(df.columns), 1)
    return rows, cols


def _get_worksheet_grid_properties(spreadsheet, sheet_id):
    """シートの現在の行数・列数を返す"""
    metadata = spreadsheet.fetch_sheet_metadata()
    for sheet in metadata.get("sheets", []):
        if sheet["properties"]["sheetId"] == sheet_id:
            grid = sheet["properties"].get("gridProperties", {})
            return grid.get("rowCount", 1), grid.get("columnCount", 1)
    return 1, 1


def _prepare_worksheet_for_formatting(spreadsheet, worksheet, df):
    """
    データ行数に合わせてシートをリサイズし、
    前回実行分の書式・入力規則・条件付き書式をクリアする。
    """
    if df.empty:
        return

    sheet_id = worksheet.id
    target_rows, target_cols = _worksheet_size_for_df(df)
    current_rows, current_cols = _get_worksheet_grid_properties(spreadsheet, sheet_id)
    clear_rows = max(current_rows, target_rows)
    clear_cols = max(current_cols, target_cols)

    _clear_worksheet_conditional_format_rules(spreadsheet, sheet_id)

    requests = [
        {
            "updateCells": {
                "range": {
                    "sheetId": sheet_id,
                    "startRowIndex": 0,
                    "endRowIndex": clear_rows,
                    "startColumnIndex": 0,
                    "endColumnIndex": clear_cols,
                },
                "fields": "userEnteredFormat,dataValidation",
            }
        },
        {
            "updateSheetProperties": {
                "properties": {
                    "sheetId": sheet_id,
                    "gridProperties": {
                        "rowCount": target_rows,
                        "columnCount": target_cols,
                    },
                },
                "fields": "gridProperties.rowCount,gridProperties.columnCount",
            }
        },
    ]
    spreadsheet.batch_update({"requests": requests})


def _apply_list_dropdown(spreadsheet, worksheet, df, column_label, options, log_message):
    """指定列にプルダウン（入力規則）を設定する"""
    if df.empty:
        return
    if column_label not in df.columns:
        print(f"⚠️ {column_label} 列が見つからないためプルダウン設定をスキップしました")
        return

    sheet_id = worksheet.id
    col_idx = list(df.columns).index(column_label)
    end_row = _get_data_row_end_index(df)

    spreadsheet.batch_update({
        "requests": [{
            "setDataValidation": {
                "range": {
                    "sheetId": sheet_id,
                    "startRowIndex": 1,
                    "endRowIndex": end_row,
                    "startColumnIndex": col_idx,
                    "endColumnIndex": col_idx + 1,
                },
                "rule": {
                    "condition": {
                        "type": "ONE_OF_LIST",
                        "values": [
                            {"userEnteredValue": option}
                            for option in options
                        ],
                    },
                    "showCustomUi": True,
                    "strict": False,
                },
            }
        }]
    })
    print(log_message)


def _apply_keiyaku_syuryo_dropdown(spreadsheet, worksheet, df):
    """契約終了理由列にプルダウン（入力規則）を設定する"""
    _apply_list_dropdown(
        spreadsheet,
        worksheet,
        df,
        KEIYAKU_SYURYO_LABEL,
        KEIYAKU_SYURYO_DROPDOWN_OPTIONS,
        "✓ 契約終了理由列にプルダウンを設定しました",
    )


def _apply_joken_henko_dropdown(spreadsheet, worksheet, df):
    """条件変更の有無列にプルダウン（入力規則）を設定する"""
    _apply_list_dropdown(
        spreadsheet,
        worksheet,
        df,
        JOKEN_HENKO_LABEL,
        JOKEN_HENKO_DROPDOWN_OPTIONS,
        "✓ 条件変更の有無列にプルダウンを設定しました",
    )


def _apply_checkbox_columns(spreadsheet, worksheet, df, column_labels):
    """指定列にチェックボックス（入力規則）を設定する"""
    if df.empty:
        return

    sheet_id = worksheet.id
    end_row = _get_data_row_end_index(df)
    requests = []

    for column_label in column_labels:
        if column_label not in df.columns:
            print(f"⚠️ {column_label} 列が見つからないためチェックボックス設定をスキップしました")
            continue
        col_idx = list(df.columns).index(column_label)
        requests.append({
            "setDataValidation": {
                "range": {
                    "sheetId": sheet_id,
                    "startRowIndex": 1,
                    "endRowIndex": end_row,
                    "startColumnIndex": col_idx,
                    "endColumnIndex": col_idx + 1,
                },
                "rule": {
                    "condition": {"type": "BOOLEAN"},
                    "showCustomUi": True,
                    "strict": True,
                },
            }
        })

    if requests:
        spreadsheet.batch_update({"requests": requests})
        print("✓ チェックボックス列を設定しました")


_HEADER_WHITE = {"red": 1, "green": 1, "blue": 1}
_HEADER_GRAY = {"red": 0.4, "green": 0.4, "blue": 0.4}
_HEADER_DARK = {"red": 0.2627451, "green": 0.2627451, "blue": 0.2627451}
_HEADER_RED = {"red": 1}
_HEADER_BLUE = {"red": 0.06666667, "green": 0.33333334, "blue": 0.8}
_HEADER_PINK = {"red": 0.7607843, "green": 0.48235294, "blue": 0.627451}
_HEADER_ORANGE = {"red": 1, "green": 0.6}
_HEADER_NAME_BLUE = {"blue": 1}


def _header_style(label):
    """参考シートのヘッダ色に合わせた書式を返す。"""
    multiline = "\n" in str(label)
    wrap = "WRAP" if multiline else "OVERFLOW_CELL"
    styles = {
        "技術者名": (_HEADER_NAME_BLUE, 10, "LEFT", "OVERFLOW_CELL", None),
        "上企業名": (_HEADER_GRAY, 10, "LEFT", "OVERFLOW_CELL", None),
        "上担当": (_HEADER_GRAY, 10, "LEFT", "OVERFLOW_CELL", None),
        "下企業名": (_HEADER_GRAY, 10, "LEFT", "OVERFLOW_CELL", None),
        "下担当": (_HEADER_GRAY, 10, "LEFT", "OVERFLOW_CELL", None),
        ARARI_COLUMN_LABEL: (_HEADER_DARK, 10, "LEFT", "OVERFLOW_CELL", None),
        UE_EIASHI_STOP_LABEL: (_HEADER_RED, 9, "CENTER", "WRAP", 132),
        SHITA_EIASHI_STOP_LABEL: (_HEADER_RED, 9, "CENTER", "WRAP", 132),
        UE_KOUSHIN_ISHI_LABEL: (_HEADER_GRAY, 10, "LEFT", "WRAP", None),
        SHITA_KOUSHIN_ISHI_LABEL: (_HEADER_GRAY, 10, "LEFT", "WRAP", None),
        "確定期間": (_HEADER_RED, 8, "LEFT", "WRAP", None),
        KEIYAKU_SYURYO_LABEL: (_HEADER_BLUE, 10, "CENTER", "WRAP", None),
        TAJIAN_KEIYAKU_SYURYO_LABEL: (_HEADER_BLUE, 10, "CENTER", "WRAP", None),
        JOKEN_HENKO_LABEL: (_HEADER_PINK, 10, "LEFT", "WRAP", None),
        KEIZOKU_MIKOMI_CHECK_LABEL: (_HEADER_PINK, 10, "LEFT", "WRAP", None),
        YOJITSU_CHECK_LABEL: (_HEADER_PINK, 8, "CENTER", "WRAP", None),
        GETSUMATSU_JOKYO_CHECK_LABEL: (_HEADER_ORANGE, 8, "CENTER", "WRAP", None),
    }
    background, size, align, wrap, width = styles.get(
        label, (_HEADER_GRAY, 10, "LEFT", wrap, None)
    )
    cell_format = {
        "backgroundColor": background,
        "horizontalAlignment": align,
        "verticalAlignment": "MIDDLE",
        "wrapStrategy": wrap,
        "textFormat": {
            "foregroundColor": _HEADER_WHITE,
            "bold": True,
            "fontFamily": "Arial",
            "fontSize": size,
        },
    }
    return cell_format, width


def _apply_header_design(spreadsheet, worksheet, df):
    """1行目のヘッダを、参考シートの配色に合わせる。"""
    if df.empty:
        return

    sheet_id = worksheet.id
    requests = [
        {
            "updateDimensionProperties": {
                "range": {
                    "sheetId": sheet_id,
                    "dimension": "ROWS",
                    "startIndex": 0,
                    "endIndex": 1,
                },
                "properties": {"pixelSize": 58},
                "fields": "pixelSize",
            }
        }
    ]
    for col_idx, label in enumerate(df.columns):
        cell_format, width = _header_style(label)
        requests.append({
            "repeatCell": {
                "range": {
                    "sheetId": sheet_id,
                    "startRowIndex": 0,
                    "endRowIndex": 1,
                    "startColumnIndex": col_idx,
                    "endColumnIndex": col_idx + 1,
                },
                "cell": {"userEnteredFormat": cell_format},
                "fields": "userEnteredFormat(backgroundColor,horizontalAlignment,verticalAlignment,wrapStrategy,textFormat)",
            }
        })
        if width:
            requests.append({
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": col_idx,
                        "endIndex": col_idx + 1,
                    },
                    "properties": {"pixelSize": width},
                    "fields": "pixelSize",
                }
            })
    spreadsheet.batch_update({"requests": requests})


def _apply_worksheet_formatting(spreadsheet, worksheet, df):
    _prepare_worksheet_for_formatting(spreadsheet, worksheet, df)
    _apply_koushin_ishi_background(spreadsheet, worksheet, df)
    _apply_keiyaku_syuryo_dropdown(spreadsheet, worksheet, df)
    _apply_joken_henko_dropdown(spreadsheet, worksheet, df)
    _apply_checkbox_columns(
        spreadsheet,
        worksheet,
        df,
        [UE_EIASHI_STOP_LABEL, SHITA_EIASHI_STOP_LABEL, *MANUAL_CHECKBOX_COLUMN_LABELS],
    )
    _apply_conditional_formatting(spreadsheet, worksheet, df)
    _apply_header_design(spreadsheet, worksheet, df)


def _write_dataframe_to_worksheet(worksheet, df):
    data_list = [df.columns.tolist()] + df.values.tolist()
    worksheet.update("A1", data_list)


def setup_google_sheets():
    """
    Google Sheetsの認証とセットアップ
    """
    try:
        print("🔐 Google Sheets認証中...")
        # Google認証
        auth.authenticate_user()

        # 認証情報を取得
        creds, _ = default()

        # Google Sheets API クライアントを作成
        gc = gspread.authorize(creds)
        print("✓ Google Sheets認証完了")
        return gc

    except Exception as e:
        print(f"❌ Google Sheets認証エラー: {e}")
        print("💡 Google Colabで認証を再実行してください")
        raise

def create_or_update_spreadsheet(gc, spreadsheet_name, df, worksheet_name="Sheet1"):
    """
    スプレッドシートを作成または更新

    Args:
        gc: Google Sheets クライアント
        spreadsheet_name: スプレッドシート名
        df: 出力するDataFrame
        worksheet_name: ワークシート名

    Returns:
        str: スプレッドシートのURL
    """
    try:
        # 既存のスプレッドシートを検索
        try:
            spreadsheet = gc.open(spreadsheet_name)
            print(f"既存のスプレッドシート '{spreadsheet_name}' を使用します")
        except:
            # 新規作成
            spreadsheet = gc.create(spreadsheet_name)
            print(f"新しいスプレッドシート '{spreadsheet_name}' を作成しました")

        # ワークシートを取得または作成
        try:
            worksheet = spreadsheet.worksheet(worksheet_name)
        except:
            rows, cols = _worksheet_size_for_df(df)
            worksheet = spreadsheet.add_worksheet(title=worksheet_name, rows=rows, cols=cols)

        # 既存データをクリア
        worksheet.clear()

        # データを一括更新
        _write_dataframe_to_worksheet(worksheet, df)
        _apply_worksheet_formatting(spreadsheet, worksheet, df)

        # スプレッドシートを共有可能に設定
        spreadsheet.share('', perm_type='anyone', role='reader')

        return spreadsheet.url

    except Exception as e:
        print(f"スプレッドシート作成/更新エラー: {e}")
        return None

def update_existing_spreadsheet(gc, spreadsheet_id, df, worksheet_name="Sheet1"):
    """
    指定されたスプレッドシートIDのスプレッドシートを更新

    Args:
        gc: Google Sheets クライアント
        spreadsheet_id: スプレッドシートID
        df: 出力するDataFrame
        worksheet_name: ワークシート名

    Returns:
        str: スプレッドシートのURL
    """
    try:
        # 指定されたIDのスプレッドシートを開く
        spreadsheet = gc.open_by_key(spreadsheet_id)
        print(f"📋 指定されたスプレッドシート（ID: {spreadsheet_id}）を開きました")

        # 既存のワークシート一覧を取得
        existing_worksheets = [ws.title for ws in spreadsheet.worksheets()]
        print(f"📄 既存のワークシート: {existing_worksheets}")

        # ワークシートを取得または作成
        if worksheet_name in existing_worksheets:
            worksheet = spreadsheet.worksheet(worksheet_name)
            print(f"✅ 既存のワークシート '{worksheet_name}' を更新します")
        else:
            rows, cols = _worksheet_size_for_df(df)
            worksheet = spreadsheet.add_worksheet(title=worksheet_name, rows=rows, cols=cols)
            print(f"🆕 新しいワークシート '{worksheet_name}' を作成しました")

        # 既存データをクリア
        worksheet.clear()
        print("🧹 既存データをクリアしました")

        # データを一括更新
        _write_dataframe_to_worksheet(worksheet, df)
        _apply_worksheet_formatting(spreadsheet, worksheet, df)
        print(f"📊 データを更新しました（{len(df)}行 × {len(df.columns)}列）")

        # スプレッドシートのURLを構築（特定のシートを指定）
        worksheet_id = worksheet.id
        spreadsheet_url = f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}/edit#gid={worksheet_id}"

        return spreadsheet_url

    except Exception as e:
        print(f"❌ スプレッドシート更新エラー: {e}")
        return None

def main():
    """
    メイン処理 - kado__c の条件一致データを取得し、指定されたスプレッドシートに出力
    グローバル変数sheetnameを使用してワークシート名を指定
    """
    global sheetname

    print("=== Salesforce kado__cデータ取得・スプレッドシート出力 ===")

    spreadsheet_id = OUTPUT_SPREADSHEET_ID
    print(f"📝 出力先: {OUTPUT_SPREADSHEET_URL}")

    sheetname, target_year, target_month = _resolve_sheetname_config()
    print(f"📄 出力先ワークシート名: {sheetname}")

    # 1. Salesforce接続
    print("\n1. Salesforceに接続中...")
    try:
        sf = setup_salesforce_connection()
        print("✓ Salesforce接続完了")
    except Exception as e:
        print(f"❌ Salesforce接続に失敗しました: {e}")
        return None, None

    # 2. kado__c データを取得
    print(
        "\n2. kado__c データを取得中..."
        f" (条件: {ENTYOKAKUNINNENGETSU_FIELD}="
        f"{_format_entyokakunin_nengetsu(target_year, target_month)}"
        f" かつ {KADONENGETSU_FIELD}="
        f"{_format_entyokakunin_nengetsu(target_year, target_month)})"
    )
    df = get_salesforce_report_data(sf, target_year=target_year, target_month=target_month)

    if df.empty:
        print("❌ 取得データが空です。条件に一致するデータがあるか、項目権限を確認してください。")
        return None, None

    print(f"✓ {len(df)}行のデータを取得しました")
    print(f"✓ カラム数: {len(df.columns)}")
    print("\nデータサンプル:")
    print(df.head())

    # 3. Google Sheets設定
    print("\n3. Google Sheetsに接続中...")
    try:
        gc = setup_google_sheets()
        print("✓ Google Sheets接続完了")
    except Exception as e:
        print(f"❌ Google Sheets接続に失敗しました: {e}")
        return None, df

    # 4. 指定されたスプレッドシートに出力
    print(f"\n4. ワークシート '{sheetname}' に出力中...")
    url = update_existing_spreadsheet(gc, spreadsheet_id, df, sheetname)

    if url:
        print(f"✓ スプレッドシート出力完了")
        print(f"🔗 スプレッドシートURL: {url}")
        return url, df
    else:
        print("❌ スプレッドシート出力に失敗しました。")
        return None, df

# 実行用関数（特定のレポートIDを指定する場合）
def run_specific_report(report_id, spreadsheet_name=None):
    """
    互換用関数。引数 report_id は使用せず、kado__c の条件一致データを取得して出力。

    Args:
        report_id: 互換用の未使用引数
        spreadsheet_name: 出力先スプレッドシート名（省略時は自動生成）
    """
    print("=== kado__c 条件一致データ取得実行 ===")

    # Salesforce接続
    sf = setup_salesforce_connection()

    # データ取得
    df = get_salesforce_report_data(sf)

    if df.empty:
        print("取得データが空です。")
        return

    print(f"✓ {len(df)}行のデータを取得しました")

    # Google Sheets設定
    gc = setup_google_sheets()

    # スプレッドシート名設定
    if not spreadsheet_name:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        spreadsheet_name = f"Salesforce_Report_{timestamp}"

    # スプレッドシートに出力
    url = create_or_update_spreadsheet(gc, spreadsheet_name, df)

    if url:
        print(f"✓ スプレッドシート出力完了")
        print(f"スプレッドシートURL: {url}")
        return url, df
    else:
        print("スプレッドシート出力に失敗しました。")
        return None, df

print("🔍 設定チェック:")
print(f"   - _sheetname_input: '{_sheetname_input}' (型: {type(_sheetname_input)})")

resolved_sheetname, resolved_year, resolved_month = _resolve_sheetname_config()

print("====================================")
print("📄 設定情報:")
print(f"   ワークシート名: '{resolved_sheetname}'")
print(f"   対象年月: {resolved_year}年{resolved_month}月")
print(
    f"   取得条件: {ENTYOKAKUNINNENGETSU_FIELD} と {KADONENGETSU_FIELD} が上記年月と一致"
)
print("====================================")

# メイン処理を実行
url, data = main()

if url and data is not None:
    from IPython.display import Markdown, display

    print(f"\n🎉 処理完了!")
    print(f"📊 取得データ: {len(data)}行 × {len(data.columns)}列")
    print(f"📄 更新したタブ: {sheetname}")
    display(Markdown(
        "## 出力先スプレッドシート\n\n"
        f"[延長確認シートを開く]({OUTPUT_SPREADSHEET_URL})\n\n"
        f"{OUTPUT_SPREADSHEET_URL}\n\n"
        f"更新したタブ: `{sheetname}`"
    ))
else:
    print("\n❌ 処理に失敗しました。")
    print("💡 以下を確認してください:")
    print("   - Salesforce認証情報が正しく設定されているか")
    print("   - kado__c の項目参照権限があるか")
    print("   - 条件に一致するデータが存在するか")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.7/191.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 2.7 MB/s eta 0:00:00
🔍 設定チェック:
   - _sheetname_input: '2026-10-01' (型: <class 'str'>)
ℹ️  日付フォーム '2026-10-01' → シート名=202610, 対象=2026年10月
📄 設定情報:
   ワークシート名: '202610'
   対象年月: 2026年10月
   取得条件: entyokakuninnengetsu__c と kadonengetsu__c が上記年月と一致
=== Salesforce kado__cデータ取得・スプレッドシート出力 ===
📝 出力先スプレッドシートID: 13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk
ℹ️  日付フォーム '2026-10-01' → シート名=202610, 対象=2026年10月
📄 出力先ワークシート名: 202610

1. Salesforceに接続中...
🔐 アクセストークンを取得中...
✓ アクセストークン取得完了
🌐 接続先: https://rosca.my.salesforce.com
✓ Salesforce接続オブジェクト作成完了
✓ Salesforce接続完了

2. kado__c データを取得中... (条件: entyokakuninnengetsu__c=2026年10月 かつ kadonengetsu__c=2026年10月)
📅 対象年月: 2026年10月（2026年10月）
🔍 SOQL: SELECT Name,gizyutusyamei__c,uekigyomei__c,uetanto__c,shitakigyomei__c,shitatanto__c,arari__c,Field1__c,uekoushinishi__c,shitakoushinishi__c,kauteikikan__c,keiyakusyuryoriyu__c,oy

/tmp/ipykernel_491/4034587795.py:905: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update("A1", data_list)


✓ ㊤更新意思・㊦更新意思列に薄水色の背景を適用しました（54行）
✓ 契約終了理由列にプルダウンを設定しました
✓ 条件変更の有無列にプルダウンを設定しました
✓ チェックボックス列を設定しました
✓ 条件付き書式を適用しました（54行）
📊 データを更新しました（54行 × 22列）
✓ スプレッドシート出力完了
🔗 スプレッドシートURL: https://docs.google.com/spreadsheets/d/13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk/edit#gid=1932392527

🎉 処理完了!
📊 取得データ: 54行 × 22列
📄 出力先ワークシート: 202610
🔗 スプレッドシートURL: https://docs.google.com/spreadsheets/d/13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk/edit#gid=1932392527
